# FinGuard Fraud Detection Pipeline
## Notebook 00 — Environment Setup

Runs once, manually, before the first pipeline execution. Creates Unity Catalog
schemas, the pipeline control (watermark) table, and verifies source files exist in unity catalog volume and runs end-to-end environment health check. Not part of the scheduled workflow.

## Imports

In [0]:
from datetime import datetime

print(f"Spark version   : {spark.version}")
print(f"Setup started   : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} AEST")

## Create Catalog and Schemas

In [0]:
CATALOG = "finguard"

schemas = {
    "raw":        "Raw source files — Unity Catalog Volume",
    "bronze":     "Bronze layer — raw Delta tables with audit columns",
    "silver":     "Silver layer — cleaned and enriched Delta tables",
    "gold":       "Gold layer — business-ready fraud feature tables",
    "monitoring": "Pipeline monitoring — DQ results, control table, dead-letter",
}

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {CATALOG}
    COMMENT 'FinGuard fraud detection pipeline — AU banking'
""")
print(f"✓ Catalog: {CATALOG}")

for schema, comment in schemas.items():
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}
        COMMENT '{comment}'
    """)
    print(f"✓ Schema : {CATALOG}.{schema}")

## Create Pipeline Control Table

Tracks the watermark (`last_processed_at`) per pipeline so each run only
processes records newer than the last successful run.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.monitoring.pipeline_control (
        pipeline_name        STRING    NOT NULL COMMENT 'Name of the pipeline',
        last_processed_at    TIMESTAMP NOT NULL COMMENT 'Watermark — last successful run timestamp',
        last_batch_id        STRING    COMMENT 'Batch ID of the last successful run',
        last_rows_processed  LONG      COMMENT 'Number of rows processed in last run',
        status               STRING    COMMENT 'last run status: success / failed',
        updated_at           TIMESTAMP COMMENT 'When this control record was last updated'
    )
    USING DELTA
    COMMENT 'Pipeline watermark control table — tracks last successful run per pipeline'
""")
print(f"✓ Control table: {CATALOG}.monitoring.pipeline_control")

pipelines = [
    "finguard_bronze_ingestion",
    "finguard_silver_transformation",
    "finguard_gold_feature_engineering",
]

existing = [
    row["pipeline_name"]
    for row in spark.sql(
        f"SELECT pipeline_name FROM {CATALOG}.monitoring.pipeline_control"
    ).collect()
]

for pipeline in pipelines:
    if pipeline not in existing:
        spark.sql(f"""
            INSERT INTO {CATALOG}.monitoring.pipeline_control VALUES (
                '{pipeline}',
                TIMESTAMP '2023-07-01 00:00:00',
                'initial_seed',
                0,
                'success',
                current_timestamp()
            )
        """)
        print(f"✓ Seeded watermark: {pipeline} → 2023-07-01")
    else:
        print(f"  Skipped (exists): {pipeline}")

## Verify Source Files

In [0]:
VOLUME_BASE = f"/Volumes/{CATALOG}/raw/source_files"

required_files = ["transactions.csv", "customers.csv", "merchants.csv"]

print(f"Checking Volume: {VOLUME_BASE}")

volume_files = [f.name for f in dbutils.fs.ls(VOLUME_BASE)]
all_present = True
for filename in required_files:
    if filename in volume_files:
        count = spark.read.option("header", "true").csv(f"{VOLUME_BASE}/{filename}").count()
        print(f"  ✓ {filename:<30} {count:>10,} rows")
    else:
        print(f"  ✗ {filename:<30} NOT FOUND")
        all_present = False

if not all_present:
    raise FileNotFoundError(
        f"Missing files in {VOLUME_BASE}. Upload all 3 CSVs before running the pipeline."
    )
print("\n✓ All source files verified")

## Environment Health Check

In [0]:
print("═" * 55)
print("  Environment Health Check")
print("═" * 55)
print(f"  Spark version : {spark.version}")
print(f"  Catalog       : {CATALOG}\n")

print("  Schemas:")
existing_schemas = [row["databaseName"] for row in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()]
for schema in schemas.keys():
    status = "✓" if schema in existing_schemas else "✗"
    print(f"    {status} {CATALOG}.{schema}")

print("\n  Control table:")
control_count = spark.sql(
    f"SELECT COUNT(*) as cnt FROM {CATALOG}.monitoring.pipeline_control"
).collect()[0]["cnt"]
print(f"    ✓ {CATALOG}.monitoring.pipeline_control ({control_count} pipelines registered)")

print("\n  Source files:")
for filename in required_files:
    print(f"    ✓ {VOLUME_BASE}/{filename}")

print("\n  Status: READY TO RUN PIPELINE")
print("  Next  : Run notebooks/pipeline/ in order, or trigger finguard_pipeline_workflow")
print("═" * 55)